# EcomPolice Data Quality and Feature Review

This notebook is a viewer and exploration layer for the preprocessing pipeline. The reusable logic lives in `src/`; this notebook loads the generated outputs, summarizes the key findings, and gives you a place to inspect tables interactively.

Run the pipeline from the project root before using this notebook:

```bash
python src/run_pipeline.py
```

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 30)

PROJECT_ROOT

ModuleNotFoundError: No module named 'pandas'

## Load Generated Outputs

These files are generated by `src/run_pipeline.py`. The raw CSV remains untouched in `data/raw/`.

In [ ]:
paths = {
    "cleaned_orders": PROJECT_ROOT / "data" / "interim" / "cleaned_orders.csv",
    "order_features": PROJECT_ROOT / "data" / "processed" / "order_features.csv",
    "pre_return_features": PROJECT_ROOT / "data" / "processed" / "pre_return_features.csv",
    "category_return_analysis": PROJECT_ROOT / "data" / "processed" / "category_return_analysis.csv",
    "payment_return_analysis": PROJECT_ROOT / "data" / "processed" / "payment_return_analysis.csv",
    "shipping_return_analysis": PROJECT_ROOT / "data" / "processed" / "shipping_return_analysis.csv",
    "return_reason_analysis": PROJECT_ROOT / "data" / "processed" / "return_reason_analysis.csv",
}

missing_outputs = [name for name, path in paths.items() if not path.exists()]
if missing_outputs:
    raise FileNotFoundError(
        "Missing generated outputs: " + ", ".join(missing_outputs) + 
        ". Run `python src/run_pipeline.py` from the project root first."
    )

cleaned_orders = pd.read_csv(paths["cleaned_orders"])
order_features = pd.read_csv(paths["order_features"])
pre_return_features = pd.read_csv(paths["pre_return_features"])
category_return_analysis = pd.read_csv(paths["category_return_analysis"])
payment_return_analysis = pd.read_csv(paths["payment_return_analysis"])
shipping_return_analysis = pd.read_csv(paths["shipping_return_analysis"])
return_reason_analysis = pd.read_csv(paths["return_reason_analysis"])

pd.DataFrame(
    [{"output": name, "path": str(path.relative_to(PROJECT_ROOT)), "rows": pd.read_csv(path).shape[0], "columns": pd.read_csv(path).shape[1]} for name, path in paths.items()]
)

## Dataset Snapshot

In [ ]:
cleaned_orders.head()

In [ ]:
summary = {
    "rows": len(cleaned_orders),
    "columns_cleaned_orders": cleaned_orders.shape[1],
    "columns_order_features": order_features.shape[1],
    "returned_orders": int((cleaned_orders["Return_Status"] == "Returned").sum()),
    "not_returned_orders": int((cleaned_orders["Return_Status"] == "Not Returned").sum()),
    "overall_return_rate": cleaned_orders["Return_Status_Binary"].mean(),
    "invalid_return_chronology": int(cleaned_orders["Invalid_Return_Chronology"].sum()),
    "valid_return_durations": int(((cleaned_orders["Return_Status"] == "Returned") & cleaned_orders["Validated_Days_To_Return"].notna()).sum()),
    "unique_users": cleaned_orders["User_ID"].nunique(dropna=False),
    "users_with_repeated_orders": int((cleaned_orders["User_ID"].value_counts(dropna=False) > 1).sum()),
}

pd.Series(summary)

## Missingness and Duplicate Checks

In [ ]:
missingness = cleaned_orders.isna().sum().rename("missing_count").to_frame()
missingness["missing_percent"] = (missingness["missing_count"] / len(cleaned_orders) * 100).round(2)
missingness.sort_values("missing_count", ascending=False)

In [ ]:
duplicate_checks = pd.Series({
    "exact_duplicate_rows": int(cleaned_orders.duplicated().sum()),
    "duplicate_Order_ID": int(cleaned_orders["Order_ID"].duplicated().sum()),
    "duplicate_Product_ID": int(cleaned_orders["Product_ID"].duplicated().sum()),
    "duplicate_User_ID": int(cleaned_orders["User_ID"].duplicated().sum()),
})

duplicate_checks

## Return Duration Quality

Negative date-derived durations are preserved as data-quality issues. They are not converted to absolute values.

In [ ]:
duration_quality = pd.Series({
    "returned_orders": int((cleaned_orders["Return_Status"] == "Returned").sum()),
    "invalid_return_chronology": int(cleaned_orders["Invalid_Return_Chronology"].sum()),
    "validated_duration_missing_for_returned": int(((cleaned_orders["Return_Status"] == "Returned") & cleaned_orders["Validated_Days_To_Return"].isna()).sum()),
    "stored_duration_mismatches": int(cleaned_orders["Stored_Duration_Mismatch"].sum()),
})

duration_quality

In [ ]:
cleaned_orders.loc[
    cleaned_orders["Invalid_Return_Chronology"],
    ["Order_ID", "Order_Date", "Return_Date", "Days_to_Return", "Calculated_Days_To_Return", "Validated_Days_To_Return", "Invalid_Return_Chronology"],
].head(10)

## Engineered Feature Preview

In [ ]:
feature_columns = [
    "Order_ID",
    "Product_Category",
    "Product_Price",
    "Order_Quantity",
    "Order_Value",
    "Order_Month",
    "Order_Quarter",
    "Order_DayOfWeek",
    "Order_IsWeekend",
    "Return_Status_Binary",
    "Validated_Days_To_Return",
    "Invalid_Return_Chronology",
]

order_features[feature_columns].head(10)

## Aggregate Return Analysis

These aggregate tables use `Return_Status`. Keep them as analysis outputs. They should not be merged back into row-level predictive features without a leakage-safe training design.

In [ ]:
category_return_analysis

In [ ]:
payment_return_analysis

In [ ]:
shipping_return_analysis

In [ ]:
return_reason_analysis

## Pre-Return Candidate Features

This table excludes return outcome, return date, return reason, stored return duration, calculated return durations, and post-return quality flags. It is a future return-prediction input table, not a fraud-detection dataset.

In [ ]:
pre_return_features.head()

In [ ]:
post_return_fields = {
    "Return_Status",
    "Return_Status_Binary",
    "Return_Date",
    "Return_Reason",
    "Days_to_Return",
    "Calculated_Days_To_Return",
    "Validated_Days_To_Return",
    "Invalid_Return_Chronology",
}

sorted(post_return_fields.intersection(pre_return_features.columns))

## Limitations

- The dataset has no verified fraud label.
- `Return_Status` is a return outcome, not a fraud label.
- Every observed `User_ID` appears once in the current file, so meaningful customer-history behavior cannot be derived from this dataset alone.
- Invalid return chronology is documented and flagged, not repaired automatically.